# CFG construction from disassembly (Section II-C in the main paper)
- The following code is used to extract the raw control flow graphs from the assembly code of the binary files under asm_sample directory.
- The raw control flow graphs are stored in the json format under the raw_cfg directory.
- We only inclue five example asm files from Big-15 dataset to show the code is functional. But the code is scalable to process a large number of files. 
- The CFG construction code is built from MCBG codebase which is available here https://github.com/Bowen-n/MCBG. 


In [2]:

import os
from asm import *

#this is the path of the input asm files 
big2015_dir = '../data/examples/inputs/asm_sample'
#this is the output path with raw cfgs
store_dir = '../data/examples/outputs/raw_cfg'
#cfg is stored in json 
file_format = 'json'

with open('empty_code.err', 'r') as f:
    empty_code_ids = f.read().split('\n')

count = 0
file_list = os.listdir(big2015_dir)
file_list = list(filter(lambda x: '.asm' in x, file_list))
for filepath in file_list:
    count += 1
    # if '.asm' not in filepath:
    #     continue

    print('{}/{} File: {}. Info: '.format(count, len(file_list), filepath), end='')

    binary_id = filepath.split('.')[0]
    if binary_id in empty_code_ids:
        print('Empty code.')
        continue

    store_path = os.path.join(store_dir, '{}.{}'.format(binary_id, file_format))
    if os.path.exists(store_path):
        print('Already parsed.')
        continue

    parser = AsmParser(directory=big2015_dir, binary_id=binary_id)
    success = parser.parse()
    if success:
        parser.store_blocks(store_path, fformat=file_format)
        print('Success.')
    else:
        print('Empty code or block after parse.')



1/5 File: 0aVxkvmflEizUBG2rMT4.asm. Info: Success.
2/5 File: 0AguvpOCcaf2myVDYFGb.asm. Info: Success.
3/5 File: 0aSTGBVRXeJhx5OcpsgC.asm. Info: Success.
4/5 File: 0AnoOZDNbPXIr2MRBSCJ.asm. Info: Success.
5/5 File: 0aVNj3qFgEZI6Akf4Kuv.asm. Info: Success.


# Print the CFG in pretty json 

In [3]:
filepath = "../data/examples/outputs/raw_cfg/0aVxkvmflEizUBG2rMT4.json"

with open(filepath, 'r') as f:
    cfg = json.load(f)
   

addr_to_id = dict() 
current_node_id = -1
block_list = list()

for addr, block in cfg.items(): # addr is 'str
    current_node_id += 1
    addr_to_id[addr] = current_node_id

    # get tokenized opcode sequence as node attributes
    block_list.append(block)
    
#print the first basic block of the CFG
print(json.dumps(block_list[0], indent=4))

edge_list = list()
for addr, block in cfg.items(): # addr is `str`
    start_nid = addr_to_id[addr]
    for out_edge in block['out_edge_list']:
        end_nid = addr_to_id[str(out_edge)]

        ## edge_index
        edge_list.append([str(start_nid), str(end_nid)])

#print all the edges in the CFG
print("edge list is \n {}".format(edge_list))




{
    "start_addr": 4199792,
    "end_addr": 4199805,
    "insn_list": [
        {
            "address": 4199792,
            "opcode": "push",
            "operands": [
                "ebp"
            ],
            "next_addr": 4199793
        },
        {
            "address": 4199793,
            "opcode": "mov",
            "operands": [
                "ebp",
                "esp"
            ],
            "next_addr": 4199795
        },
        {
            "address": 4199795,
            "opcode": "sub",
            "operands": [
                "esp",
                "8"
            ],
            "next_addr": 4199798
        },
        {
            "address": 4199798,
            "opcode": "mov",
            "operands": [
                "[ebp+-8]",
                "1"
            ],
            "next_addr": 4199805
        },
        {
            "address": 4199805,
            "opcode": "mov",
            "operands": [
                "[ebp+-4]",
                "0"